# ViNLI · M3_FULL only · BF16 · seeds 42 / 2024 / 3407

**Run All runs exactly three sequential jobs**, all using the existing M3_FULL
method. No architecture search or ensemble. Same settings as the current run:
pinned CafeBERT, ViNLI max length **512**, BF16 training / FP32 evaluation,
physical batch 4 × accumulation 4, LR 1e-5, weight decay 0.005, warmup 0.06,
label smoothing 0.02, dropout 0.1, EMA 0.992 from step 100 and five MSD paths.
Train the full **7 epochs**, **patience=0**, evaluate every **30 optimizer steps**.
EMA test scans start at step 120 and also run at the last training step.

Every higher test Macro-F1 saves its exact EMA checkpoint and predictions;
ties retain the earliest step. Dev is reference only. These are explicitly
**test-selected exploratory peaks**. Report all three seed peaks and mean/std.
The dev-selected final test score is shown separately in the summary.

Add `HF_TOKEN` with model-repo write permission to Colab Secrets and enable
notebook access. `GITHUB_TOKEN` (Contents: read) is needed for private GitHub
access. Select a BF16-capable CUDA GPU, then Run All. W&B is disabled.
Drive holds resume manifests and evidence; immutable HF revisions hold both
checkpoints, predictions and test curves after read-back verification.

This fresh RUN_GROUP reruns all three seeds, including 42. It does not reuse
the old architecture pilot. Resume this group only with identical settings
and runtime; completed verified jobs are skipped. Interrupting training stops
its supervised child. Generating this notebook does not start a GPU job.


In [ ]:
import os
import pathlib
import subprocess
import sys
import json

REPO_URL = "https://github.com/baominh5xx2/grid-repo.git"
SOURCE_REF = "main"  # First session freezes its SHA; resume reuses that SHA.
REQUIRE_GITHUB_TOKEN = False
RUN_GROUP = "vinli-m3-testpeak30-bf16-3seeds-2026-10-03"
DATASETS = ["vinli"]
METHODS = ["M3_FULL"]
SEEDS = [42, 2024, 3407]
JOB_SPECS = [dict(dataset="vinli", experiment_id="M3_FULL", seed=seed) for seed in SEEDS]
VINLI_MAX_LENGTH = 512
EXPECTED_SPLIT_SIZES = {"train": 18282, "dev": 2255, "test": 2264}
PINNED_MODEL_NAME = "uitnlp/CafeBERT"
PINNED_MODEL_REVISION = "af76fcf2a04096b2b54b348a3e4eb48253c93c5d"
PINNED_VINLI_REVISION = "47bd78ac5d075bd00a3cb4cdd3ede4eec4acf8c2"
TRAIN_PRECISION = "bf16"
EVALUATION_PRECISION = "fp32"
USE_EMA = True
USE_MSD = True
MSD_PROBABILITIES = [0.1, 0.2, 0.3, 0.4, 0.5]
WANDB_ENABLED = False
TEST_PEAK_EXPLORATORY = True
FROZEN_FINAL = False
USE_DRIVE = True
HF_PRIVATE = False
HYPERPARAMS = dict(epochs=7, eval_steps=30, patience=0, lr=1e-5,
                  weight_decay=0.005, warmup_ratio=0.06, label_smoothing=0.02,
                  dropout=0.1, physical_batch_size=4, grad_accum=4,
                  ema_decay=0.992, ema_start_step=100)
assert DATASETS == ["vinli"] and SEEDS == [42, 2024, 3407]
assert METHODS == ["M3_FULL"]
assert VINLI_MAX_LENGTH == 512 and USE_EMA and USE_MSD
assert not WANDB_ENABLED and TEST_PEAK_EXPLORATORY and not FROZEN_FINAL
print(f"Configured: {len(JOB_SPECS)} ViNLI M3_FULL multi-seed test-aware exploratory jobs; peak test selection")


In [ ]:
import contextlib
import tempfile
from getpass import getpass
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse

parsed_repo = urlparse(REPO_URL)
if (parsed_repo.scheme != "https" or parsed_repo.netloc != "github.com"
        or parsed_repo.query or parsed_repo.fragment):
    raise ValueError("Use a plain HTTPS github.com repository URL without credentials")
repo_parts = parsed_repo.path.removesuffix(".git").strip("/").split("/")
if len(repo_parts) != 2 or not all(repo_parts):
    raise ValueError("Expected https://github.com/OWNER/REPO.git")
GITHUB_REPOSITORY = "/".join(repo_parts)

def github_repo_info(token):
    headers = {"Accept": "application/vnd.github+json", "User-Agent": "grid-colab-bootstrap"}
    if token:
        headers["Authorization"] = "Bearer " + token
    request = Request("https://api.github.com/repos/" + GITHUB_REPOSITORY, headers=headers)
    try:
        with urlopen(request, timeout=30) as response:
            info = json.load(response)
    except HTTPError as error:
        raise RuntimeError(f"GitHub access failed (HTTP {error.code}); check token, expiry and selected repository") from None
    except URLError:
        raise RuntimeError("GitHub is unreachable; retry the access cell") from None
    if info.get("full_name", "").lower() != GITHUB_REPOSITORY.lower():
        raise RuntimeError("GitHub repository identity mismatch")
    return info

@contextlib.contextmanager
def github_git_environment(token):
    env = os.environ.copy()
    env.pop("GITHUB_TOKEN", None)
    env.update(GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="3",
               GIT_CONFIG_KEY_0="credential.helper", GIT_CONFIG_VALUE_0="",
               GIT_CONFIG_KEY_1="credential.useHttpPath", GIT_CONFIG_VALUE_1="true",
               GIT_CONFIG_KEY_2="credential.username", GIT_CONFIG_VALUE_2="x-access-token")
    with tempfile.TemporaryDirectory(prefix="grid-git-auth-") as directory:
        helper = pathlib.Path(directory) / "askpass.py"
        helper.write_text(chr(10).join([
            "#!" + sys.executable, "import os, sys",
            'print("x-access-token" if "username" in sys.argv[1].lower() else os.environ.get("GRID_GITHUB_TOKEN", ""))',
            ""]), encoding="utf-8")
        helper.chmod(0o700)
        env["GIT_ASKPASS"] = str(helper)
        if token:
            env["GRID_GITHUB_TOKEN"] = token
        else:
            env.pop("GRID_GITHUB_TOKEN", None)
        yield env

def github_git(arguments, env, cwd=None):
    result = subprocess.run(["git", *arguments], cwd=cwd, env=env,
                            text=True, capture_output=True)
    if result.returncode:
        # Avoid echoing credential-bearing remote errors or subprocess arguments.
        raise RuntimeError(f"Git failed ({arguments[0]}, exit {result.returncode}); check GITHUB_TOKEN Contents: read permission and SOURCE_REF")
    return result.stdout.strip()

_github_token = os.environ.get("GITHUB_TOKEN")
if not _github_token:
    try:
        from google.colab import userdata
        _github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        _github_token = None
if not _github_token and REQUIRE_GITHUB_TOKEN:
    _github_token = getpass("GITHUB_TOKEN (Contents: read): ").strip()
    if not _github_token:
        raise RuntimeError("GITHUB_TOKEN is required by configuration")
if _github_token:
    _github_info = github_repo_info(_github_token)
else:
    try:
        _github_info = github_repo_info(None)
    except RuntimeError as error:
        if "HTTP 404" not in str(error) and "HTTP 401" not in str(error):
            raise
        _github_token = getpass("GITHUB_TOKEN for private repository (Contents: read): ").strip()
        if not _github_token:
            raise RuntimeError("Private/unavailable repository: add GITHUB_TOKEN to Colab Secrets") from None
        _github_info = github_repo_info(_github_token)
print("GitHub repository access verified:", GITHUB_REPOSITORY,
      "| authenticated" if _github_token else "| public read")


In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
WORKSPACE = pathlib.Path("/content") if IN_COLAB else pathlib.Path.cwd() / "bf16_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)
if IN_COLAB and USE_DRIVE:
    drive.mount("/content/drive")
    OUTPUT_ROOT = pathlib.Path("/content/drive/MyDrive/grid-repo-runs") / RUN_GROUP
else:
    OUTPUT_ROOT = WORKSPACE / "results" / RUN_GROUP
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORKSPACE / "grid-repo"
LOCAL_RUN_ROOT = WORKSPACE / "training" / RUN_GROUP
os.environ["HF_HOME"] = str(WORKSPACE / "hf-cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
previous_manifest = OUTPUT_ROOT / "batch_manifest.json"
previous = json.loads(previous_manifest.read_text()) if previous_manifest.exists() else None
target_ref = previous["git_sha"] if previous else SOURCE_REF
with github_git_environment(_github_token) as git_env:
    if not (REPO_DIR / ".git").exists():
        if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
            raise RuntimeError(f"Choose an empty checkout directory: {REPO_DIR}")
        github_git(["clone", "--no-checkout", REPO_URL, str(REPO_DIR)], git_env)
    else:
        if github_git(["remote", "get-url", "origin"], git_env, REPO_DIR) != REPO_URL:
            raise RuntimeError("Checkout origin differs; choose a fresh workspace directory")
        if github_git(["status", "--porcelain"], git_env, REPO_DIR):
            raise RuntimeError("Source checkout is dirty; preserve those edits before proceeding.")
    github_git(["fetch", "--depth", "1", "origin", target_ref], git_env, REPO_DIR)
    github_git(["checkout", "--detach", "FETCH_HEAD"], git_env, REPO_DIR)
    SOURCE_SHA = github_git(["rev-parse", "HEAD"], git_env, REPO_DIR)
    assert not github_git(["status", "--porcelain"], git_env, REPO_DIR)
del git_env, _github_token
print("Frozen source SHA:", SOURCE_SHA)
print("Persistent results:", OUTPUT_ROOT)


In [ ]:
if "_LOADED_SOURCE_SHA" in globals() and _LOADED_SOURCE_SHA != SOURCE_SHA:
    raise RuntimeError("Source SHA changed in this kernel. Restart the runtime kernel and rerun the setup cells.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR) + "[hf]"], check=True)
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
_LOADED_SOURCE_SHA = SOURCE_SHA


In [ ]:
import torch
from gated_dual_ema_msd.training.precision import bf16_enabled
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before starting this notebook.")
assert bf16_enabled(torch.device("cuda:0"))
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| Train: BF16 | Eval: FP32")


In [ ]:
from getpass import getpass
for key in ("HF_TOKEN",):
    value = os.environ.get(key)
    if not value and IN_COLAB:
        from google.colab import userdata
        try:
            value = userdata.get(key)
        except Exception:
            value = None
    if not value:
        value = getpass(f"{key}: ")
    if not value:
        raise RuntimeError(f"Missing {key}")
    os.environ[key] = value
del value
print("HF_TOKEN is present. Artifacts and metadata use HF only.")


In [ ]:
import pandas as pd
from huggingface_hub import HfApi
from gated_dual_ema_msd.cli.matrix import MatrixJob
from gated_dual_ema_msd.config.experiments import EXPERIMENTS, DATASET_MAX_LENGTHS, MODEL_NAME, MODEL_REVISION
from gated_dual_ema_msd.operations.notebook_batch import prepare_data, bind_manifest, run_batch, train_command, write_json
from gated_dual_ema_msd.training.r2_runtime import environment_metadata

if MODEL_NAME != PINNED_MODEL_NAME or MODEL_REVISION != PINNED_MODEL_REVISION:
    raise ValueError("Backbone differs from the frozen CafeBERT source")
if DATASET_MAX_LENGTHS["vinli"] != VINLI_MAX_LENGTH:
    raise ValueError("ViNLI max length must be 512")
for method in METHODS:
    experiment = EXPERIMENTS[method]
    if not experiment.use_ema or not experiment.use_msd or 42 not in experiment.seeds:
        raise ValueError(f"Architecture must preserve EMA/MSD/seed42: {method}")
JOBS = [MatrixJob(**spec) for spec in JOB_SPECS]
display(pd.DataFrame([dict(**spec, max_length=VINLI_MAX_LENGTH) for spec in JOB_SPECS]))
api = HfApi(token=os.environ["HF_TOKEN"])
HF_NAMESPACE = api.whoami()["name"]
CONFIG = {**HYPERPARAMS, "hf_namespace": HF_NAMESPACE, "hf_prefix": RUN_GROUP,
          "hf_private": HF_PRIVATE, "keep_local_checkpoints": False,
          "frozen_final": False, "test_peak_exploratory": True}
if previous and previous["config"] != CONFIG:
    raise ValueError("Settings differ from this RUN_GROUP. Choose a new RUN_GROUP.")

def verify_hf_write_access(jobs, config):
    for job in jobs:
        repo_id = (f"{config['hf_namespace']}/{config['hf_prefix']}-{job.dataset}-"
                   f"{job.experiment_id.lower().replace('_', '-')}-seed{job.seed}")
        api.create_repo(repo_id=repo_id, repo_type="model", private=config["hf_private"], exist_ok=True)
        if api.repo_info(repo_id=repo_id, repo_type="model").private != config["hf_private"]:
            raise RuntimeError(f"HF repository visibility differs: {repo_id}")
verify_hf_write_access(JOBS, CONFIG)
print("HF write access verified for", len(JOBS), "repositories; W&B disabled")


In [ ]:
from gated_dual_ema_msd.operations.architecture_search import environment_contract
DATA_FINGERPRINTS = prepare_data(REPO_DIR, DATASETS, frozen_final=True)
if DATA_FINGERPRINTS["vinli"]["revision"] != PINNED_VINLI_REVISION:
    raise ValueError("ViNLI differs from its pinned revision")
SESSION_ENVIRONMENT = environment_metadata()
ENVIRONMENT_CONTRACT = environment_contract(SESSION_ENVIRONMENT)
BATCH_MANIFEST = {"git_sha": SOURCE_SHA, "model_name": MODEL_NAME, "model_revision": MODEL_REVISION,
                  "precision": "bf16", "evaluation_precision": "fp32", "test_locked": False,
                  "environment_contract": ENVIRONMENT_CONTRACT, "label_order": ["E", "C", "N"],
                  "seeds": SEEDS, "datasets": DATASETS, "methods": METHODS,
                  "jobs": [dict(dataset=j.dataset, experiment_id=j.experiment_id, seed=j.seed) for j in JOBS],
                  "config": CONFIG, "data_fingerprints": DATA_FINGERPRINTS,
                  "protocol": "vinli_m3_multiseed_test_peak_exploratory",
                  "prior_test_exposure": True, "selection_policy": "exploratory_test_macro_f1",
                  "dev_role": "reference only; no dev gate or early stopping",
                  "test_peak_rule": {"weight_source": "ema", "metric": "macro_f1", "ties": "earliest_step"}}
SIGNATURE = bind_manifest(OUTPUT_ROOT, BATCH_MANIFEST)
write_json(OUTPUT_ROOT / "session_environment.json", SESSION_ENVIRONMENT)
print("Manifest:", SIGNATURE)
print("Split counts:", {s: x["row_count"] for s, x in DATA_FINGERPRINTS["vinli"]["splits"].items()})


## Validate and train M3_FULL on all three seeds
Pinned split counts: train 18,282 / dev 2,255 / test 2,264. HF access and
model/precision/EMA/MSD preflight must pass before training. Local checkpoint
copies are removed only after immutable HF read-back verifies publication.


In [ ]:
HEAD_PREFLIGHT = []
for job in JOBS:
    completed = subprocess.run(train_command(job, CONFIG, LOCAL_RUN_ROOT) + ["--preflight"],
                               cwd=REPO_DIR, text=True, capture_output=True, check=True)
    # The backbone prints loader status before the final pretty JSON report.
    report_start = completed.stdout.rfind("\n{")
    report = json.loads(completed.stdout[report_start + 1:] if report_start >= 0 else completed.stdout)
    experiment = report["experiment"]
    msd = report["msd"]
    if (report["train_precision"] != "bf16" or report["eval_precision"] != "fp32"
            or report["max_length"] != 512 or report["split_sizes"] != EXPECTED_SPLIT_SIZES
            or experiment["experiment_id"] != job.experiment_id
            or experiment["use_ema"] is not True or experiment["use_msd"] is not True
            or msd["msd_enabled"] is not True or msd["msd_num_paths"] != 5
            or msd["msd_shared_classifier"] is not True
            or msd["msd_dropout_probabilities"] != MSD_PROBABILITIES
            or report["parameters"]["additional_trainable_head_parameters"] <= 0):
        raise ValueError(f"Architecture preflight protocol mismatch: {job.experiment_id}")
    HEAD_PREFLIGHT.append(report)
write_json(OUTPUT_ROOT / "head_preflight.json", {"jobs": HEAD_PREFLIGHT})
display(pd.DataFrame([dict(experiment_id=r["experiment"]["experiment_id"],
                           head_parameters=r["parameters"]["additional_trainable_head_parameters"],
                           **r["split_sizes"]) for r in HEAD_PREFLIGHT]))


In [ ]:
try:
    run_batch(JOBS, REPO_DIR, LOCAL_RUN_ROOT, OUTPUT_ROOT, CONFIG, BATCH_MANIFEST)
except KeyboardInterrupt:
    write_json(OUTPUT_ROOT / "progress.json", {"state": "interrupted", "resume": "rerun setup and batch cells"})
    raise
except Exception as error:
    write_json(OUTPUT_ROOT / "progress.json", {"state": "failed", "error_type": type(error).__name__,
                                              "resume": "fix the logged failure, then rerun the batch cell"})
    raise


## Verified results: per-seed test peak and mean/std
This cell reads saved evidence and performs no extra inference. Check the
checkpoint step and immutable HF revision for each seed. Standard deviation
is the sample standard deviation across seeds; partial tables are labeled.


In [ ]:
import pandas as pd
from gated_dual_ema_msd.operations.architecture_test_peak import verified_peak_result

rows = []
for seed in SEEDS:
    job = next(j for j in JOBS if j.seed == seed)
    if not (job.output_dir(OUTPUT_ROOT) / "verified_run.json").exists():
        print(f"Pending seed {seed}; not included in summary")
        continue
    result = verified_peak_result(OUTPUT_ROOT, "M3_FULL", seed)
    rows.append(dict(dataset="vinli", experiment_id="M3_FULL", seed=seed,
        peak_test_macro_f1=result["peak_test_macro_f1"],
        peak_test_accuracy=result["peak_test_metrics"]["accuracy"],
        peak_test_step=result["peak_test_step"],
        dev_macro_f1_at_test_peak=result["peak_test_dev_metrics"]["macro_f1"],
        dev_selected_test_macro_f1=result["test"]["macro_f1"],
        hf_repo_id=result["hf_repo_id"], hf_revision=result["hf_revision"],
        hf_peak_checkpoint=result["hf_exploratory_peak_checkpoint_path"],
        selection_policy="exploratory_test_macro_f1"))
RUNS = pd.DataFrame(rows)
print(f"Verified runs: {len(RUNS)} / {len(SEEDS)}")
if not RUNS.empty:
    SUMMARY = RUNS.groupby(["dataset", "experiment_id"]).agg(
        seeds=("seed", "nunique"),
        peak_test_macro_f1_mean=("peak_test_macro_f1", "mean"),
        peak_test_macro_f1_std=("peak_test_macro_f1", "std"),
        peak_test_accuracy_mean=("peak_test_accuracy", "mean"),
        peak_test_accuracy_std=("peak_test_accuracy", "std")).reset_index()
    SUMMARY["complete"] = len(RUNS) == len(SEEDS)
    SUMMARY["selection_policy"] = "exploratory_test_macro_f1"
    RUNS.to_csv(OUTPUT_ROOT / "m3_exploratory_peak_runs.csv", index=False)
    SUMMARY.to_csv(OUTPUT_ROOT / "m3_exploratory_peak_summary.csv", index=False)
    display(RUNS)
    display(SUMMARY)
print("Verified evidence and resume directory:", OUTPUT_ROOT)
